# HW2 - Costa Rica Household Poverty Level Prediction
**Machine Learning II | IE University**

**Evaluation Metric:** Macro F1-Score  
**Task:** Multi-class classification — 4 poverty levels (1=extreme poverty → 4=non-vulnerable)  
**Architecture:** Stacked Generalization — LightGBM · XGBoost · CatBoost → Logistic Regression meta-learner

---
### Strategy Overview
1. **Data Quality Fix** — align household-level targets across all members  
2. **Feature Engineering** — composite poverty indices from raw indicators  
3. **Imbalance Handling** — SMOTE applied inside each fold (no leakage) + class weights  
4. **Level-0 Base Models** — LightGBM, XGBoost, CatBoost with 5-fold StratifiedKFold CV → OOF probability predictions  
5. **Level-1 Meta-Learner** — Logistic Regression trained on the OOF meta-feature matrix (n_train × 12)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', 50)
np.random.seed(42)

print('All imports successful')

## 2. Load Data

In [ ]:
DATA_PATH = 'costa-rican-household-poverty-prediction/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')
print(f'\nTarget distribution (train):')
target_counts = train['Target'].value_counts().sort_index()
target_labels = {1: 'Extreme poverty', 2: 'Moderate poverty', 3: 'Vulnerable', 4: 'Non-vulnerable'}
for cls, cnt in target_counts.items():
    pct = cnt / len(train) * 100
    print(f'  Class {cls} ({target_labels[cls]}): {cnt:>5} rows ({pct:.1f}%)')

## 3. Data Quality Fix — Household Target Consistency

The `Target` is a household-level attribute, but some households have members with inconsistent targets.
We correct this by assigning the head-of-household's (`parentesco1=1`) target to all household members.

In [ ]:
# Identify households with inconsistent targets
household_targets = train.groupby('idhogar')['Target'].nunique()
inconsistent = household_targets[household_targets > 1]
print(f'Households with inconsistent targets: {len(inconsistent)}')

# Fix: use head-of-household target for all members
heads = train[train['parentesco1'] == 1][['idhogar', 'Target']].rename(columns={'Target': 'head_target'})
train = train.merge(heads, on='idhogar', how='left')

# Fill missing head_target (households without a clear head) with original target
train['head_target'] = train['head_target'].fillna(train['Target'])
train['Target'] = train['head_target'].astype(int)
train.drop(columns=['head_target'], inplace=True)

# Verify fix
remaining = train.groupby('idhogar')['Target'].nunique()
print(f'Households with inconsistent targets after fix: {(remaining > 1).sum()}')

## 4. Missing Value Analysis & Imputation

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values (train):')
for col, cnt in missing.items():
    print(f'  {col}: {cnt} ({cnt/len(train)*100:.1f}%)')

In [ ]:
train['v2a1'] = train['v2a1'].fillna(0)
test['v2a1']  = test['v2a1'].fillna(0)

train['v18q1'] = train['v18q1'].fillna(0)
test['v18q1']  = test['v18q1'].fillna(0)

train['rez_esc'] = train['rez_esc'].fillna(0)
test['rez_esc']  = test['rez_esc'].fillna(0)

meaneduc_median = train['meaneduc'].median()
train['meaneduc'] = train['meaneduc'].fillna(meaneduc_median)
test['meaneduc']  = test['meaneduc'].fillna(meaneduc_median)

# SQBmeaned is meaneduc squared — impute consistently after filling meaneduc
train['SQBmeaned'] = train['SQBmeaned'].fillna(train['meaneduc'] ** 2)
test['SQBmeaned']  = test['SQBmeaned'].fillna(test['meaneduc'] ** 2)

for col in ['edjefe', 'edjefa']:
    train[col] = train[col].replace({'no': 0, 'yes': 1}).astype(float).fillna(0)
    test[col]  = test[col].replace({'no': 0, 'yes': 1}).astype(float).fillna(0)

train['dependency'] = train['dependency'].replace({'yes': 1, 'no': 0}).astype(float)
test['dependency']  = test['dependency'].replace({'yes': 1, 'no': 0}).astype(float)
train['dependency'] = train['dependency'].fillna(train['dependency'].median())
test['dependency']  = test['dependency'].fillna(train['dependency'].median())

print('Missing values remaining (train):', train.isnull().sum().sum())
print('Missing values remaining (test): ', test.isnull().sum().sum())

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
ax = axes[0]
colors = ['#d62728', '#ff7f0e', '#ffbb78', '#2ca02c']
labels = ['1\nExtreme\nPoverty', '2\nModerate\nPoverty', '3\nVulnerable', '4\nNon-\nVulnerable']
bars = ax.bar(labels, target_counts.values, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val}\n({val/len(train)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_title('Target Class Distribution\n(Train Set)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Households')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Overcrowding by poverty level
ax = axes[1]
for cls, color in zip([1,2,3,4], colors):
    subset = train[train['Target'] == cls]['overcrowding']
    ax.hist(subset, bins=20, alpha=0.6, label=target_labels[cls], color=color, density=True)
ax.set_title('Overcrowding Distribution by Poverty Level', fontsize=12, fontweight='bold')
ax.set_xlabel('Persons per Room')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Education vs poverty
ax = axes[0]
train.boxplot(column='meaneduc', by='Target', ax=ax,
              boxprops=dict(color='steelblue'),
              medianprops=dict(color='red', linewidth=2))
ax.set_title('Average Education by Poverty Level', fontsize=12, fontweight='bold')
ax.set_xlabel('Poverty Level (1=Extreme, 4=Non-Vulnerable)')
ax.set_ylabel('Mean Years of Education')
plt.sca(ax)
plt.title('')

# Appliance ownership vs poverty  
ax = axes[1]
appliances = ['refrig', 'computer', 'television', 'mobilephone', 'v18q']
appliance_by_class = train.groupby('Target')[appliances].mean()
appliance_by_class.T.plot(kind='bar', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Appliance Ownership Rate by Poverty Level', fontsize=12, fontweight='bold')
ax.set_xlabel('Appliance')
ax.set_ylabel('Ownership Rate')
ax.set_xticklabels(['Refrigerator', 'Computer', 'TV', 'Mobile', 'Tablet'], rotation=20)
ax.legend(title='Poverty Level', labels=[target_labels[i] for i in [1,2,3,4]], fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eda_education_appliances.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Engineering

We create composite indices that capture multi-dimensional poverty concepts beyond individual binary indicators.

In [ ]:
def engineer_features(df):
    df = df.copy()

    # --- Infrastructure Quality Index ---
    # Combine wall, roof and floor quality into a single composite score (0-9)
    # epared: 1=bad, 2=regular, 3=good | etecho: 1=bad, 2=regular, 3=good | eviv: 1=bad, 2=regular, 3=good
    df['wall_quality']   = df['epared1']*1 + df['epared2']*2 + df['epared3']*3
    df['roof_quality']   = df['etecho1']*1 + df['etecho2']*2 + df['etecho3']*3
    df['floor_quality']  = df['eviv1']*1   + df['eviv2']*2   + df['eviv3']*3
    df['infra_index']    = df['wall_quality'] + df['roof_quality'] + df['floor_quality']

    # --- Asset / Wealth Index ---
    df['asset_index'] = (
        df['refrig'] + df['computer'] + df['television'] +
        df['mobilephone'] + df['v18q'] + (df['qmobilephone'] > 1).astype(int)
    )

    # --- Sanitation & Services Index ---
    # Higher is better
    df['sanitation_ok']  = df['sanitario2'] + df['sanitario3']  # connected to sewer or septic
    df['water_ok']       = df['abastaguadentro'].astype(int)     # water inside dwelling
    df['electricity_ok'] = df['public'].astype(int)             # public electricity grid
    df['services_index'] = df['sanitation_ok'] + df['water_ok'] + df['electricity_ok']

    # --- Housing Density ---
    df['rooms_per_person']   = df['rooms']    / (df['hhsize'] + 1)
    df['bedrooms_per_person']= df['bedrooms'] / (df['hhsize'] + 1)

    # --- Education Indicators ---
    df['max_educ_head']   = df[['edjefe', 'edjefa']].max(axis=1)
    df['educ_gap']        = df['meaneduc'] - df['escolari']  # individual vs household avg

    # --- Household Composition ---
    df['child_ratio']     = df['hogar_nin']   / (df['hogar_total'] + 1)
    df['elderly_ratio']   = df['hogar_mayor'] / (df['hogar_total'] + 1)
    df['working_age_ratio']= df['hogar_adul'] / (df['hogar_total'] + 1)

    # --- Rent burden (only for renters) ---
    df['monthly_rent'] = df['v2a1']  # already imputed to 0 for owners

    # --- Overcrowding flag ---
    df['severe_overcrowding'] = (df['overcrowding'] > 3).astype(int)

    # --- Poverty Risk Score (weighted composite) ---
    # Combines infrastructure, assets, services, education into a single risk score
    # Lower score = more risk
    df['poverty_risk_score'] = (
        df['infra_index']    * 0.3 +
        df['asset_index']    * 0.25 +
        df['services_index'] * 0.2 +
        df['meaneduc']       * 0.25
    )

    return df

train = engineer_features(train)
test  = engineer_features(test)

new_features = [
    'infra_index', 'asset_index', 'services_index',
    'rooms_per_person', 'bedrooms_per_person',
    'max_educ_head', 'educ_gap', 'child_ratio',
    'elderly_ratio', 'working_age_ratio',
    'severe_overcrowding', 'poverty_risk_score'
]
print(f'New features created: {len(new_features)}')
print(train[new_features].head(3))

## 7. Prepare Features for Modeling

We use only the **head of household** rows for training since Target is household-level. This avoids pseudo-replication.

In [ ]:
# Drop non-predictive columns
DROP_COLS = ['Id', 'idhogar', 'Target']

# Use head of household for training to avoid row duplication within households
train_heads = train[train['parentesco1'] == 1].copy()
print(f'Training rows (heads of household only): {len(train_heads)}')

# Features present in both train and test
test_cols  = set(test.columns)
train_cols = set(train_heads.columns) - set(DROP_COLS)
feature_cols = sorted(list(train_cols & test_cols))

X = train_heads[feature_cols].astype(float)
y = train_heads['Target'].astype(int)
X_test_final = test[feature_cols].astype(float)

print(f'Features used: {len(feature_cols)}')
print(f'Target distribution (heads only):')
for cls, cnt in y.value_counts().sort_index().items():
    print(f'  Class {cls} ({target_labels[cls]}): {cnt} ({cnt/len(y)*100:.1f}%)')

## 8. Class Imbalance — SMOTE

The Target is heavily skewed toward Class 4. SMOTE is applied **inside each CV fold** — never on the full dataset — so synthetic samples never leak into validation or test sets. Base models also use class weighting as a second layer of protection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

y_counts_before = y.value_counts().sort_index()
axes[0].bar([target_labels[i] for i in y_counts_before.index],
            y_counts_before.values, color=colors, edgecolor='black')
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

_X_sm, _y_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X.fillna(0), y)
y_counts_after = pd.Series(_y_sm).value_counts().sort_index()
axes[1].bar([target_labels[i] for i in y_counts_after.index],
            y_counts_after.values, color=colors, edgecolor='black')
axes[1].set_title('After SMOTE (illustration)', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Class Distribution: Before vs After SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Before: {len(X)} rows  |  After SMOTE: {len(_X_sm)} rows')
del _X_sm, _y_sm

## 9. Stacked Generalization — Level-0 Base Models (OOF)

**Architecture (Wolpert 1992):**
- **Level 0**: LightGBM, XGBoost, CatBoost trained with 5-fold StratifiedKFold. Each generates OOF probability predictions (4 classes), covering every training row once without leakage.
- **Level 1**: Logistic Regression meta-learner on the OOF matrix (n_train × 12).
- **SMOTE inside the fold**: applied only on the training portion, never the held-out portion.

```
Train set → Fold 1-5 (StratifiedKFold)
    SMOTE(train_fold) → LightGBM → OOF probs (×4)
                      → XGBoost  → OOF probs (×4)
                      → CatBoost → OOF probs (×4)
OOF matrix (n_train × 12) → Logistic Regression meta-learner
```

In [ ]:
N_FOLDS   = 5
N_CLASSES = 4
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X_np      = X.fillna(0).values
y_shifted = y.values - 1          # 0-indexed: 0,1,2,3
X_test_np = X_test_final.fillna(0).values

oof_lgb = np.zeros((len(X_np), N_CLASSES))
oof_xgb = np.zeros((len(X_np), N_CLASSES))
oof_cat = np.zeros((len(X_np), N_CLASSES))

test_lgb = np.zeros((len(X_test_np), N_CLASSES))
test_xgb = np.zeros((len(X_test_np), N_CLASSES))
test_cat = np.zeros((len(X_test_np), N_CLASSES))

lgb_params = dict(
    objective='multiclass', num_class=4, metric='multi_logloss',
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.1,
    class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1
)
xgb_params = dict(
    objective='multi:softprob', num_class=4, eval_metric='mlogloss',
    n_estimators=400, learning_rate=0.05, max_depth=6,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0, n_jobs=-1
)
cat_params = dict(
    iterations=400, learning_rate=0.05, depth=6,
    loss_function='MultiClass', random_seed=42, verbose=0,
    class_weights=[4, 2, 2, 1]
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_shifted)):
    print(f'Fold {fold+1}/{N_FOLDS}', end='  ')
    X_tr, X_val_f = X_np[tr_idx], X_np[val_idx]
    y_tr, y_val_f = y_shifted[tr_idx], y_shifted[val_idx]

    X_tr_sm, y_tr_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_tr, y_tr)

    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr_sm, y_tr_sm)
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val_f)
    test_lgb        += m_lgb.predict_proba(X_test_np) / N_FOLDS

    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr_sm, y_tr_sm)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val_f)
    test_xgb        += m_xgb.predict_proba(X_test_np) / N_FOLDS

    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(X_tr_sm, y_tr_sm)
    oof_cat[val_idx] = m_cat.predict_proba(X_val_f)
    test_cat        += m_cat.predict_proba(X_test_np) / N_FOLDS

    f1s = [f1_score(y_val_f, np.argmax(oof_lgb[val_idx], 1), average='macro'),
           f1_score(y_val_f, np.argmax(oof_xgb[val_idx], 1), average='macro'),
           f1_score(y_val_f, np.argmax(oof_cat[val_idx], 1), average='macro')]
    print(f'LGB {f1s[0]:.4f}  XGB {f1s[1]:.4f}  CAT {f1s[2]:.4f}')

print('\nOOF Macro F1:')
for name, oof in [('LightGBM', oof_lgb), ('XGBoost', oof_xgb), ('CatBoost', oof_cat)]:
    print(f'  {name:10s}: {f1_score(y_shifted, np.argmax(oof, 1), average="macro"):.4f}')

## 10. Level-1 Meta-Learner

In [ ]:
meta_train = np.hstack([oof_lgb, oof_xgb, oof_cat])
meta_test  = np.hstack([test_lgb, test_xgb, test_cat])

print(f'meta_train: {meta_train.shape}   [LGB×4 | XGB×4 | CAT×4]')
print(f'meta_test : {meta_test.shape}')

meta_lr = LogisticRegression(
    C=1.0, solver='lbfgs', max_iter=1000,
    multi_class='multinomial', class_weight='balanced', random_state=42
)

meta_cv = cross_val_score(
    meta_lr, meta_train, y_shifted,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1
)
print(f'\nStacked Ensemble — Macro F1 (5-fold CV):')
print(f'  Folds : {meta_cv.round(4)}')
print(f'  Mean  : {meta_cv.mean():.4f} ± {meta_cv.std():.4f}')

meta_lr.fit(meta_train, y_shifted)

## 11. Evaluation — Base Models vs Stacked Ensemble

In [ ]:
meta_oof_pred = cross_val_predict(
    meta_lr, meta_train, y_shifted,
    cv=StratifiedKFold(5, shuffle=True, random_state=42)
)
print(classification_report(y_shifted, meta_oof_pred,
      target_names=[target_labels[i] for i in [1, 2, 3, 4]]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_shifted, meta_oof_pred)
ConfusionMatrixDisplay(cm, display_labels=[target_labels[i] for i in [1,2,3,4]]).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Stacked Ensemble (OOF)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

f1_per = {
    'LightGBM': f1_score(y_shifted, np.argmax(oof_lgb, 1), average=None),
    'XGBoost' : f1_score(y_shifted, np.argmax(oof_xgb, 1), average=None),
    'CatBoost': f1_score(y_shifted, np.argmax(oof_cat, 1), average=None),
    'Stack'   : f1_score(y_shifted, meta_oof_pred,          average=None),
}
bar_c = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
x, w = np.arange(4), 0.2
for i, (name, f1s) in enumerate(f1_per.items()):
    axes[1].bar(x + (i-1.5)*w, f1s, w, label=name, color=bar_c[i], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Class {i+1}\n{target_labels[i+1][:10]}' for i in range(4)], fontsize=8)
axes[1].set_ylabel('F1-Score')
axes[1].set_title('Per-Class F1: Base Models vs Stacked Ensemble', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('stacking_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('OOF Macro F1:')
for name, oof in [('LightGBM', oof_lgb), ('XGBoost', oof_xgb), ('CatBoost', oof_cat)]:
    print(f'  {name:10s}: {f1_score(y_shifted, np.argmax(oof,1), average="macro"):.4f}')
print(f'  {"Stack":10s}: {f1_score(y_shifted, meta_oof_pred, average="macro"):.4f}')

## 12. Generate Kaggle Submission

In [ ]:
y_test_pred = meta_lr.predict(meta_test) + 1   # map back to 1-4

submission = pd.DataFrame({'Id': test['Id'], 'Target': y_test_pred})
submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(f'Rows: {len(submission)}')
print('\nPredicted class distribution:')
for cls, cnt in pd.Series(y_test_pred).value_counts().sort_index().items():
    print(f'  Class {cls} ({target_labels[cls]}): {cnt} ({cnt/len(submission)*100:.1f}%)')
print(submission.head())